# oilspill — Cerulean reproduction (Colab driver)

This notebook is a **thin driver**. All logic lives in the `oilspill` package;
cells only install, mount, call functions and display results.

Cells 1–3 are the Phase 0/1 verification. Run them in order and paste the
outputs back.

| Cell | Phase | What it does |
|---|---|---|
| 1 | 0 | install + mount Drive (idempotent) |
| 2 | 0 | GPU / runtime check |
| 3 | 1 | real-data spike + overlays |
| 4+ | 2–13 | added as each phase lands |

> Attribution: SkyTruth Cerulean (cerulean.skytruth.org). 2026. CC BY-SA 4.0. 
> Contains modified Copernicus Sentinel data 2026.

## Cell 1 — install + mount

**Before running:** set `REPO_URL` to your public GitHub repo. If the repo does
not exist yet, leave it as `None` and use the Drive fallback described in the
cell (zip `src/` to `MyDrive/oilspill/code`).

Safe to re-run after a disconnect — it re-mounts and re-installs idempotently.

In [ ]:
REPO_URL = None  # e.g. "https://github.com/<you>/oilspill.git"

import os, subprocess, sys
from pathlib import Path

# --- Drive ---------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive/oilspill')
for sub in ('tiles', 'splits', 'checkpoints', 'exports', 'logs', 'status', 'cache'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print('Drive root:', ROOT)

# Bulky artifacts live on Drive so a disconnect never loses work.
os.environ['OILSPILL_DATA_DIR'] = str(ROOT)

# --- package -------------------------------------------------------------
def sh(cmd):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout or '')[-2000:])
    if r.returncode != 0:
        print('STDERR:', (r.stderr or '')[-2000:])
    return r.returncode

if REPO_URL:
    if not Path('/content/oilspill/.git').exists():
        sh(f'git clone --depth 1 {REPO_URL} /content/oilspill')
    else:
        sh('cd /content/oilspill && git pull --ff-only')
    code_dir = '/content/oilspill'
else:
    # Fallback: unzip src from Drive (MyDrive/oilspill/code/oilspill_src.zip)
    code_dir = '/content/oilspill'
    zip_path = ROOT / 'code' / 'oilspill_src.zip'
    if zip_path.exists():
        sh(f'mkdir -p {code_dir} && unzip -oq "{zip_path}" -d {code_dir}')
    else:
        raise SystemExit(
            f'Set REPO_URL, or upload the source zip to {zip_path}'
        )

sh(f'pip install -q -e {code_dir}')

import importlib
if 'oilspill' in sys.modules:
    for m in [k for k in sys.modules if k.startswith('oilspill')]:
        del sys.modules[m]
import oilspill
from oilspill import config
print('oilspill', oilspill.__version__)
print('PIXEL_DEG', config.PIXEL_DEG)
print('DATA_DIR ', config.DATA_DIR)
config.ensure_dirs()

## Cell 2 — GPU / runtime check (Phase 0 success criterion)

Phase 0 passes when a GPU is visible here. If `torch.cuda.is_available()` is
`False`, switch Runtime → Change runtime type → T4 GPU and re-run. **Do not
train on CPU** — fall back to Kaggle instead.

In [ ]:
import platform, shutil, subprocess, sys

print('python    ', sys.version.split()[0], platform.platform())
print('-' * 70)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi: NOT AVAILABLE')
print('-' * 70)
try:
    import torch
    print('torch      ', torch.__version__)
    print('cuda avail ', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('gpu        ', p.name, f'{p.total_memory/1e9:.1f} GB')
except ImportError:
    print('torch not installed yet (only needed from Phase 4)')

import rasterio, shapely, numpy
print('rasterio   ', rasterio.__version__, '| GDAL', rasterio.__gdal_version__, '| PROJ', rasterio.__proj_version__)
print('shapely    ', shapely.__version__)
print('numpy      ', numpy.__version__)

total, used, free = shutil.disk_usage('/content')
print(f'disk /content  free {free/1e9:.1f} GB of {total/1e9:.1f} GB')

## Cell 3 — Phase 1 spike (real data)

Already **passed locally 3/3** (see `reports/phase1/`). Re-running here confirms
the same result on Colab and measures Colab's read throughput, which sets the
Phase 2 scale.

Expected: 3 slicks, contrast ≈ 3.4–3.9 dB, both orbit directions, `3/3 passed`.

In [ ]:
!cd /content/oilspill && python scripts/spike_phase1.py --n-regions 3

In [ ]:
# Display the overlays produced by cell 3 (criterion (a) is a visual check).
from pathlib import Path

from IPython.display import Image, display

from oilspill import config

out = config.REPORTS_DIR / 'phase1'
overlays = sorted(out.glob('*_overlay.png'))
print(f'{len(overlays)} overlays in {out}')
for p in overlays:
    print(p.name)
    display(Image(filename=str(p), width=900))

---

## Cells 4+ — Phases 2–13

Added as each phase is implemented and verified. Planned order:

4. build dataset · 5. inspect tiles · 6. splits · 7. model · 8. train (resume) 
9. evaluate · 10. visualise · 11. save · 12. inference on a real scene 
13. ONNX export + model card · 14. optional API demo via uvicorn + tunnel